# Multimodal VAE — Evaluation

Evaluation-only notebook for the multimodal VAE with Product-of-Experts fusion.
Assumes processed data already exists (run the baseline training notebook first).

**Before running:**
1. Add your processed data dataset and set `PROC_DATA_SLUG`
2. Add your model weights dataset and set `WEIGHTS_SLUG`
3. Enable GPU

In [ ]:
!pip install umap-learn pyarrow -q

In [ ]:
# ── Configure these ──────────────────────────────────────────────────────────
PROC_DATA_SLUG = 'metabonet-processed'   # dataset containing data/processed/
WEIGHTS_SLUG   = 'multimodal-vae-weights' # dataset containing the .pt checkpoint files
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path

PROC_DIR   = Path(f'/kaggle/input/{PROC_DATA_SLUG}/data/processed')
WEIGHTS_DIR = Path(f'/kaggle/input/{WEIGHTS_SLUG}')
OUT_DIR    = Path('/kaggle/working/eval_multimodal')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Processed data: {PROC_DIR}  exists={PROC_DIR.exists()}')
print(f'Weights dir:    {WEIGHTS_DIR}  exists={WEIGHTS_DIR.exists()}')
print(f'Output:         {OUT_DIR}')

# List available checkpoints
ckpts = sorted(WEIGHTS_DIR.glob('checkpoint_epoch_*.pt'))
print(f'\nCheckpoints found: {[c.name for c in ckpts]}')
print(f'best_model.pt:     {(WEIGHTS_DIR / "best_model.pt").exists()}')

## 1. Source Code

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import json
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score, silhouette_samples
from tqdm import tqdm
import umap

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
class ConvEncoder1D(nn.Module):
    def __init__(self, input_length, channels, kernel_size=3, stride=1, padding=1, latent_dim=8):
        super().__init__()
        layers = []
        in_ch  = 1
        current_length = input_length
        for out_ch in channels:
            layers += [nn.Conv1d(in_ch, out_ch, kernel_size, stride=stride, padding=padding), nn.ReLU()]
            current_length = (current_length - kernel_size + 2 * padding) // stride + 1
            in_ch = out_ch
        self.conv_layers   = nn.Sequential(*layers)
        self.conv_out_size = in_ch * current_length
        self.fc_mu         = nn.Linear(self.conv_out_size, latent_dim)
        self.fc_logvar     = nn.Linear(self.conv_out_size, latent_dim)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = self.conv_layers(x).view(x.size(0), -1)
        return self.fc_mu(x), self.fc_logvar(x)


class ConvDecoder1D(nn.Module):
    def __init__(self, output_length, channels, latent_dim=8, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.output_length     = output_length
        self.conv_out_channels = channels[0]
        self.conv_out_length   = max(output_length // (2 ** len(channels)), 1)
        self.fc = nn.Linear(latent_dim, self.conv_out_channels * self.conv_out_length)
        layers  = []
        in_ch   = channels[0]
        for out_ch in channels[1:] + [1]:
            layers.append(nn.ConvTranspose1d(in_ch, out_ch, kernel_size, stride=stride, padding=padding))
            if out_ch != 1:
                layers.append(nn.ReLU())
            in_ch = out_ch
        self.deconv_layers = nn.Sequential(*layers)

    def forward(self, z):
        x = self.fc(z).view(z.size(0), self.conv_out_channels, self.conv_out_length)
        x = self.deconv_layers(x).squeeze(1)
        if x.size(1) > self.output_length:
            x = x[:, :self.output_length]
        elif x.size(1) < self.output_length:
            x = F.pad(x, (0, self.output_length - x.size(1)))
        return x


class MultimodalVAEWithPoE(nn.Module):
    def __init__(self, cgm_length=288, insulin_length=288, physiology_dim=5,
                 encoder_channels=None, decoder_channels=None, latent_dim=8):
        super().__init__()
        encoder_channels = encoder_channels or [16, 32, 64]
        decoder_channels = decoder_channels or [64, 32, 16]
        self.latent_dim = latent_dim

        self.cgm_encoder      = ConvEncoder1D(cgm_length,    encoder_channels, latent_dim=latent_dim)
        self.insulin_encoder  = ConvEncoder1D(insulin_length, encoder_channels, latent_dim=latent_dim)
        self.physiology_encoder = nn.Sequential(
            nn.Linear(physiology_dim, 64), nn.ReLU(),
            nn.Linear(64, 32),             nn.ReLU(),
        )
        self.physiology_mu     = nn.Linear(32, latent_dim)
        self.physiology_logvar = nn.Linear(32, latent_dim)

        self.cgm_decoder    = ConvDecoder1D(cgm_length,    decoder_channels, latent_dim=latent_dim)
        self.insulin_decoder = ConvDecoder1D(insulin_length, decoder_channels, latent_dim=latent_dim)
        self.physiology_decoder = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(),
            nn.Linear(32, 64),         nn.ReLU(),
            nn.Linear(64, physiology_dim),
        )

    def product_of_experts(self, mu_list, logvar_list):
        var_list      = [torch.exp(lv) for lv in logvar_list]
        prec_sum      = sum(1.0 / (v + 1e-8) for v in var_list)
        weighted_sum  = sum((1.0 / (v + 1e-8)) * m for m, v in zip(mu_list, var_list))
        mu_poe        = weighted_sum / (prec_sum + 1e-8)
        logvar_poe    = torch.log(1.0 / (prec_sum + 1e-8) + 1e-8)
        return mu_poe, logvar_poe

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

    def forward(self, cgm=None, insulin=None, physiology=None):
        mu_list, lv_list = [], []
        if cgm is not None:
            m, lv = self.cgm_encoder(cgm)
            mu_list.append(m); lv_list.append(lv)
        if insulin is not None:
            m, lv = self.insulin_encoder(insulin)
            mu_list.append(m); lv_list.append(lv)
        if physiology is not None:
            h  = self.physiology_encoder(physiology)
            mu_list.append(self.physiology_mu(h))
            lv_list.append(self.physiology_logvar(h))
        if not mu_list:
            raise ValueError('At least one modality must be provided')
        mu, logvar = self.product_of_experts(mu_list, lv_list)
        z = self.reparameterize(mu, logvar)
        out = {'mu': mu, 'logvar': logvar, 'z': z}
        if cgm is not None:        out['cgm_recon']        = self.cgm_decoder(z)
        if insulin is not None:    out['insulin_recon']    = self.insulin_decoder(z)
        if physiology is not None: out['physiology_recon'] = self.physiology_decoder(z)
        return out

print('Model class ready.')

In [ ]:
class MetaboNetDataset(Dataset):
    def __init__(self, processed_dir, split='test', normalize=True):
        self.split_dir = Path(processed_dir) / split
        self.normalize = normalize
        if not self.split_dir.exists():
            raise FileNotFoundError(f'Processed data not found: {self.split_dir}')
        self._load()
        self.scalers = {}
        if normalize:
            self._fit_scalers()

    def _load(self):
        self.cgm_data        = np.load(self.split_dir / 'cgm.npy',        mmap_mode='r')
        self.insulin_data    = np.load(self.split_dir / 'insulin.npy',    mmap_mode='r')
        self.physiology_data = np.load(self.split_dir / 'physiology.npy', mmap_mode='r')
        self.metadata        = pd.read_parquet(self.split_dir / 'metadata.parquet')
        self.n_samples       = len(self.cgm_data)

    def _fit_scalers(self):
        cgm_fit = np.array(self.cgm_data,        dtype=np.float32)
        ins_fit = np.array(self.insulin_data,     dtype=np.float32)
        phy_fit = np.array(self.physiology_data,  dtype=np.float32)
        phy_obs = phy_fit[~np.isnan(phy_fit).any(axis=1)]
        self.scalers['cgm']     = StandardScaler().fit(cgm_fit)
        self.scalers['insulin'] = StandardScaler().fit(ins_fit)
        if len(phy_obs):
            self.scalers['physiology'] = StandardScaler().fit(phy_obs)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        cgm = np.array(self.cgm_data[idx], dtype=np.float32)
        if self.normalize and 'cgm' in self.scalers:
            cgm = self.scalers['cgm'].transform(cgm.reshape(1, -1)).flatten()

        insulin = np.array(self.insulin_data[idx], dtype=np.float32)
        if self.normalize and 'insulin' in self.scalers:
            insulin = self.scalers['insulin'].transform(insulin.reshape(1, -1)).flatten()

        phys = np.array(self.physiology_data[idx], dtype=np.float32)
        if np.isnan(phys).any():
            phys = np.zeros(phys.shape, dtype=np.float32)
        elif self.normalize and 'physiology' in self.scalers:
            phys = self.scalers['physiology'].transform(phys.reshape(1, -1)).flatten()

        row = self.metadata.iloc[idx]
        return {
            'cgm':        torch.from_numpy(cgm),
            'insulin':    torch.from_numpy(insulin),
            'physiology': torch.from_numpy(phys),
            'patient_id': row['patient_id'],
            'date':       row['date'],
        }

print('Loading test set...')
test_set = MetaboNetDataset(PROC_DIR, split='test', normalize=True)
print(f'Test samples: {len(test_set):,}')

## 2. Load Model

Selects the best checkpoint after KL warm-up (epoch >= 10). If `training_history.json`
is not available, falls back to `best_model.pt`.

In [ ]:
# ── Architecture — match whatever was used during training ───────────────────
LATENT_DIM       = 8
ENCODER_CHANNELS = [16, 32, 64]
DECODER_CHANNELS = [64, 32, 16]
WARMUP_EPOCHS    = 10
# ─────────────────────────────────────────────────────────────────────────────

model = MultimodalVAEWithPoE(
    cgm_length       = 288,
    insulin_length   = 288,
    physiology_dim   = 5,
    encoder_channels = ENCODER_CHANNELS,
    decoder_channels = DECODER_CHANNELS,
    latent_dim       = LATENT_DIM,
)

# Pick best post-warmup checkpoint from training history if available
hist_path = WEIGHTS_DIR / 'training_history.json'
ckpts     = sorted(WEIGHTS_DIR.glob('checkpoint_epoch_*.pt'))

if hist_path.exists() and ckpts:
    with open(hist_path) as f:
        history = json.load(f)
    val_losses  = history['val_loss']
    saved_epochs = [int(p.stem.split('_')[-1]) for p in ckpts]
    candidates   = [(ep, val_losses[ep - 1]) for ep in saved_epochs if ep > WARMUP_EPOCHS]
    if candidates:
        best_epoch, best_loss = min(candidates, key=lambda x: x[1])
        weights_path = WEIGHTS_DIR / f'checkpoint_epoch_{best_epoch:03d}.pt'
        ckpt = torch.load(weights_path, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        print(f'Loaded checkpoint epoch {best_epoch}  val_loss={best_loss:.4f}')
    else:
        weights_path = WEIGHTS_DIR / 'best_model.pt'
        model.load_state_dict(torch.load(weights_path, map_location=device))
        print(f'No post-warmup checkpoints found — loaded best_model.pt')
else:
    weights_path = WEIGHTS_DIR / 'best_model.pt'
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print(f'Loaded best_model.pt')

model = model.to(device)
model.eval()
print(f'Model on device: {next(model.parameters()).device}')

## 3. Reconstruction Quality

In [ ]:
# 6 random test samples — CGM and insulin reconstructions
torch.manual_seed(0)
with torch.no_grad():
    batch = next(iter(DataLoader(test_set, batch_size=6, shuffle=True)))
    cgm      = batch['cgm'].to(device)
    insulin  = batch['insulin'].to(device)
    physio   = batch['physiology'].to(device)
    out = model(cgm=cgm, insulin=insulin, physiology=physio)

cgm_np      = cgm.cpu().numpy()
cgm_recon   = out['cgm_recon'].cpu().numpy()
ins_np      = insulin.cpu().numpy()
ins_recon   = out['insulin_recon'].cpu().numpy()
time_h      = np.arange(288) * 5 / 60

for modality, orig, recon, ylabel in [
        ('CGM',     cgm_np, cgm_recon, 'CGM (normalised)'),
        ('Insulin', ins_np, ins_recon, 'Insulin (normalised)'),
]:
    fig, axes = plt.subplots(2, 3, figsize=(15, 6))
    for i, ax in enumerate(axes.flat):
        mse = float(((orig[i] - recon[i]) ** 2).mean())
        ax.plot(time_h, orig[i],  label='original',      alpha=0.85, linewidth=1.2)
        ax.plot(time_h, recon[i], label='reconstructed', alpha=0.85, linewidth=1.2, linestyle='--')
        ax.set_title(f'Patient {batch["patient_id"][i]} | {batch["date"][i]}\nMSE={mse:.4f}')
        ax.set_xlabel('Hour of day')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
    fig.suptitle(f'{modality} reconstruction — 6 test samples', fontsize=13)
    fig.tight_layout()
    plt.savefig(OUT_DIR / f'recon_{modality.lower()}.png', dpi=100)
    plt.show()

In [ ]:
# Per-modality reconstruction MSE on full test set
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
totals = {'cgm': 0.0, 'insulin': 0.0, 'physiology': 0.0}
n = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test MSE'):
        cgm    = batch['cgm'].to(device)
        ins    = batch['insulin'].to(device)
        phys   = batch['physiology'].to(device)
        out    = model(cgm=cgm, insulin=ins, physiology=phys)
        totals['cgm']        += nn.MSELoss()(out['cgm_recon'],        cgm).item()
        totals['insulin']    += nn.MSELoss()(out['insulin_recon'],    ins).item()
        totals['physiology'] += nn.MSELoss()(out['physiology_recon'], phys).item()
        n += 1

print('Test reconstruction MSE:')
for k, v in totals.items():
    print(f'  {k:12s}: {v/n:.6f}')

## 4. Latent Space Analysis

Uses the PoE-fused `mu` (deterministic) as the latent representation — combines
information from all three modalities.

In [ ]:
# Encode full test set — PoE fused mu
all_mu, all_pids = [], []

with torch.no_grad():
    for batch in tqdm(DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2),
                      desc='Encoding'):
        out = model(
            cgm       = batch['cgm'].to(device),
            insulin   = batch['insulin'].to(device),
            physiology= batch['physiology'].to(device),
        )
        all_mu.append(out['mu'].cpu().numpy())
        all_pids.extend(batch['patient_id'])

Z    = np.vstack(all_mu)
pids = np.array(all_pids)
print(f'Latent matrix: {Z.shape}')

In [ ]:
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
Z2d = reducer.fit_transform(Z)

unique_pids = np.unique(pids)
cmap = plt.cm.get_cmap('tab20', len(unique_pids))
fig, ax = plt.subplots(figsize=(10, 8))
for i, pid in enumerate(unique_pids):
    mask = pids == pid
    ax.scatter(Z2d[mask, 0], Z2d[mask, 1], s=5, alpha=0.5, color=cmap(i), label=str(pid))
ax.set_title('Latent space UMAP — test set (coloured by patient)')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
if len(unique_pids) <= 20:
    ax.legend(markerscale=3, fontsize=8, title='Patient', bbox_to_anchor=(1.05, 1), loc='upper left')
fig.tight_layout()
plt.savefig(OUT_DIR / 'umap.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Unique patients: {len(unique_pids)}')

## 5. Proxy-Free Metrics

In [ ]:
le     = LabelEncoder()
labels = le.fit_transform(pids)

MAX_N = 5000
if len(Z) > MAX_N:
    idx      = np.random.default_rng(42).choice(len(Z), MAX_N, replace=False)
    Z_s, L_s = Z[idx], labels[idx]
else:
    Z_s, L_s = Z, labels

score       = silhouette_score(Z_s, L_s, metric='euclidean')
sil_samples = silhouette_samples(Z_s, L_s, metric='euclidean')

print(f'Silhouette score: {score:.4f}')
print(f'  >0.50  strong  |  0.25-0.50  moderate  |  <0.25  weak')

pid_labels_s   = le.inverse_transform(L_s)
df_sil         = pd.DataFrame({'patient': pid_labels_s, 'silhouette': sil_samples})
patient_scores = df_sil.groupby('patient')['silhouette'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(max(8, len(patient_scores) * 0.5), 5))
colors = ['steelblue' if s > 0 else 'tomato' for s in patient_scores.values]
ax.bar(range(len(patient_scores)), patient_scores.values, color=colors)
ax.axhline(0,     color='black',  linewidth=0.8, linestyle='--')
ax.axhline(score, color='orange', linewidth=1.5, linestyle='--', label=f'overall={score:.3f}')
ax.set_xticks(range(len(patient_scores)))
ax.set_xticklabels(patient_scores.index, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Patient'); ax.set_ylabel('Mean silhouette score')
ax.set_title('Per-patient silhouette  (blue=well separated, red=mixed)')
ax.legend()
fig.tight_layout()
plt.savefig(OUT_DIR / 'silhouette_per_patient.png', dpi=100)
plt.show()

In [ ]:
meta = test_set.metadata.reset_index(drop=True)
rng  = np.random.default_rng(42)
adj_dists, rand_dists = [], []

for pid in np.unique(pids):
    mask  = pids == pid
    idx   = np.where(mask)[0]
    order = np.argsort(meta.loc[idx, 'date'].values)
    zs    = Z[idx][order]
    if len(zs) < 2:
        continue
    for i in range(len(zs) - 1):
        adj_dists.append(np.linalg.norm(zs[i] - zs[i + 1]))
    for _ in range(len(zs) - 1):
        i, j = rng.choice(len(zs), size=2, replace=False)
        rand_dists.append(np.linalg.norm(zs[i] - zs[j]))

adj_mean  = float(np.mean(adj_dists))
rand_mean = float(np.mean(rand_dists))
print(f'Adjacent day distance:  {adj_mean:.4f}')
print(f'Random pair distance:   {rand_mean:.4f}')
print(f'Ratio (adj / random):   {adj_mean / rand_mean:.3f}  (lower = more temporally smooth)')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(adj_dists,  bins=50, alpha=0.6, label=f'Adjacent days  (mean={adj_mean:.3f})')
ax.hist(rand_dists, bins=50, alpha=0.6, label=f'Random pairs   (mean={rand_mean:.3f})')
ax.set_xlabel('Euclidean distance in latent space')
ax.set_ylabel('Count')
ax.set_title('Temporal consistency')
ax.legend()
fig.tight_layout()
plt.savefig(OUT_DIR / 'temporal_consistency.png', dpi=100)
plt.show()

In [ ]:
N_SHOW    = min(6, len(np.unique(pids)))
show_pids = np.unique(pids)[:N_SHOW]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, pid in zip(axes.flat, show_pids):
    mask  = pids == pid
    idx   = np.where(mask)[0]
    order = np.argsort(meta.loc[idx, 'date'].values)
    xy    = Z2d[idx][order]
    n     = len(xy)
    cvals = np.linspace(0, 1, n)
    for i in range(n - 1):
        ax.plot(xy[i:i+2, 0], xy[i:i+2, 1], color=plt.cm.coolwarm(cvals[i]), alpha=0.7, linewidth=1.5)
    ax.scatter(xy[:, 0], xy[:, 1], c=cvals, cmap='coolwarm', s=20, zorder=5)
    ax.scatter(*xy[0],  color='green', s=90, zorder=6, marker='^', label='start')
    ax.scatter(*xy[-1], color='red',   s=90, zorder=6, marker='s', label='end')
    ax.set_title(f'Patient {pid}  ({n} days)')
    ax.legend(fontsize=7)
fig.suptitle('Latent trajectories in UMAP space — blue=early, red=late', fontsize=13)
fig.tight_layout()
plt.savefig(OUT_DIR / 'latent_trajectories.png', dpi=100)
plt.show()